In [ ]:
import io
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns

sns.set_theme(style="whitegrid")

# 1. Load Dataset
url = "https://docs.google.com/spreadsheets/d/1-0lKSG-Gbh2ym48fhfYxPgs7URLJh-2py1CNeqzIKCU/export?format=csv"
# If reading locally: df = pd.read_csv('spread_locator_dataset.csv')
df = pd.read_csv(url)

print("=== DATA OVERVIEW ===")
print(df.info())
print("\nDescriptive Statistics:\n", df['transaction_amount'].describe())

# 2. Bernoulli and Binomial Fitting
p_success = (df['transaction_status'] == 'Success').mean()
print(f"\n[Bernoulli] P(Success): {p_success:.4f}")

n_trials = df['transaction_count'].max()
p_binomial = df['transaction_count'].mean() / n_trials
print(
    f"[Binomial] n={n_trials}, estimated p={p_binomial:.4f} (mean count:"
    f" {df['transaction_count'].mean():.2f})"
)

# 3. Poisson Fitting (Daily Transactions)
daily_counts = df.groupby('transaction_date').size()
lambda_poisson = daily_counts.mean()
print(
    f"\n[Poisson] Daily Arrival Rate lambda: {lambda_poisson:.4f} (Variance:"
    f" {daily_counts.var():.4f})"
)

# 4. Distribution Fitting for Transaction Amount
shape_ln, loc_ln, scale_ln = stats.lognorm.fit(
    df['transaction_amount'], floc=0
)
b_pareto, loc_pareto, scale_pareto = stats.pareto.fit(df['transaction_amount'])

ks_norm = stats.kstest(
    df['transaction_amount'],
    'norm',
    args=(df['transaction_amount'].mean(), df['transaction_amount'].std()),
)
ks_ln = stats.kstest(
    df['transaction_amount'], 'lognorm', args=(shape_ln, loc_ln, scale_ln)
)

print(f"\n[KS Test] Normal p-value: {ks_norm.pvalue:.4e}")
print(f"[KS Test] Log-Normal p-value: {ks_ln.pvalue:.4f}")

# 5. Q-Q Plot & Box-Cox Transform
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
stats.probplot(df['transaction_amount'], dist='norm', plot=axes[0])
axes[0].set_title("Q-Q Plot: Raw Transaction Amount")

transformed_amt, best_lambda = stats.boxcox(df['transaction_amount'])
stats.probplot(transformed_amt, dist='norm', plot=axes[1])
axes[1].set_title(f"Q-Q Plot: Box-Cox Transformed (λ={best_lambda:.4f})")
plt.tight_layout()
plt.show()

# 6. Z-Score Calculation for Rs. 5000
mean_val = df['transaction_amount'].mean()
std_val = df['transaction_amount'].std()
z_5000 = (5000 - mean_val) / std_val
prob_normal_5000 = 1 - stats.norm.cdf(z_5000)
prob_empirical_5000 = (df['transaction_amount'] > 5000).mean()
prob_lognorm_5000 = 1 - stats.lognorm.cdf(
    5000, shape_ln, loc=loc_ln, scale=scale_ln
)

print(f"\nZ-Score for Rs. 5000: {z_5000:.4f}")
print(f"P(X > 5000) [Normal Model]: {prob_normal_5000 * 100:.2f}%")
print(f"P(X > 5000) [Empirical]:    {prob_empirical_5000 * 100:.2f}%")
print(f"P(X > 5000) [Log-Normal]:   {prob_lognorm_5000 * 100:.2f}%")

# 7. PDF and CDF Plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(
    df['transaction_amount'], kde=True, stat="density", ax=axes[0], color="navy"
)
x_vals = np.linspace(
    df['transaction_amount'].min(), df['transaction_amount'].max(), 500
)
axes[0].plot(
    x_vals,
    stats.lognorm.pdf(x_vals, shape_ln, loc=loc_ln, scale=scale_ln),
    'r--',
    label='Log-Normal Fit',
)
axes[0].set_title("PDF: Transaction Amounts vs. Log-Normal Fit")
axes[0].legend()

sns.ecdfplot(
    df['transaction_amount'],
    ax=axes[1],
    color="navy",
    label="Empirical Cumulative (ECDF)",
)
axes[1].plot(
    x_vals,
    stats.lognorm.cdf(x_vals, shape_ln, loc=loc_ln, scale=scale_ln),
    'r--',
    label='Log-Normal CDF Fit',
)
axes[1].axvline(
    5000, color='darkgreen', linestyle=':', label='₹5,000 Threshold'
)
axes[1].set_title("CDF: Cumulative Distribution Function")
axes[1].legend()
plt.tight_layout()
plt.show()